In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import re
import numpy as np

# =================================================================
# ✨ 튜터 노트: 데이터셋 정보 파악하기
# =================================================================

DATASET_ID = "CocoRoF/cc-100-korean-processing"
SAMPLE_COUNT = 5 # 실습을 위해 상위 5개 샘플만 사용합니다.

print("=" * 60)
print(f"📖 데이터셋 로딩 준비: {DATASET_ID}")
print("=" * 60)

# 1. 사용 가능한 Config 목록 확인하기
try:
    configs = get_dataset_config_names(DATASET_ID)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 메인 실습을 위해 첫 번째 config를 사용합니다.
    selected_config = configs[0]
    print(f"\n🚀 사용할 Config: '{selected_config}'")

except Exception as e:
    print(f"⚠️ Config 목록 조회 중 오류가 발생했습니다. 에러: {e}")
    selected_config = None

# 2. 데이터셋 로딩 전략 수립 (Streaming vs. Standard)
dataset = None
try:
    print("\n⚙️ 데이터셋 로딩 시도: 스트리밍 모드 (Streaming=True)를 사용하여 메모리를 절약합니다.")
    # 스트리밍 모드는 데이터셋이 매우 클 때 메모리 폭발을 막아주는 마법 같은 기능입니다.
    # 이걸 사용하면 필요할 때 필요한 만큼만 데이터를 가져오죠!
    dataset = load_dataset(DATASET_ID, name=selected_config, split='train', streaming=True)
    print("✨ [성공] 스트리밍 모드 로딩에 성공했습니다! (최적의 선택!)")

except Exception as e:
    # 스트리밍 모드가 어떤 환경에서 불안정할 수 있으므로, 예외 처리를 합니다.
    print(f"\n⚠️ [경고] 스트리밍 모드 로딩에 실패했습니다. 일반 모드로 전환합니다. (에러: {e})")
    try:
        # 일반 모드로 로딩 (적은 수의 데이터만 다운로드)
        dataset = load_dataset(DATASET_ID, name=selected_config, split='train')
        print("✨ [성공] 일반 모드로 로딩에 성공했습니다.")
    except Exception as inner_e:
        print(f"🛑 데이터셋 로딩에 실패했습니다. 실행 환경을 확인해 주세요. ({inner_e})")
        exit()


# 3. 샘플 데이터 준비 (Constraint 9, 16 준수)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print(f"\n📦 {SAMPLE_COUNT}개의 샘플을 추출하여 실습을 준비합니다...")
    # 스트리밍 데이터셋은 바로 list()로 변환할 수 없기 때문에, take()를 사용합니다.
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))

print(f"✅ 준비된 샘플 개수: {len(sample_data_list)}개")


# =================================================================
# 🏆 실습 1: 데이터 탐색 및 구조 분석 (Inspection)
# =================================================================
print("\n" + "#" * 60)
print("🚀 [실습 1] 데이터 내용 엿보기: 핵심 필드 확인")
print("#" * 60)

print("✨ 친절한 튜터 코멘트: 이 데이터셋은 'text'라는 필드 하나만 가지고 있어요. 즉, 문장 그 자체를 다루는 것이 핵심입니다!")

text_samples = [sample['text'] for sample in sample_data_list]

if text_samples:
    print("\n🔍 추출된 텍스트 샘플 5개:")
    for i, text in enumerate(text_samples):
        print(f"  [Sample {i+1}] ✂️ 텍스트 일부: {text[:50]}...")
else:
    print("😭 샘플 데이터를 가져오지 못했어요. 데이터셋 로드를 다시 확인해 주세요!")


# =================================================================
# 🏆 실습 2: 데이터 정량 분석 - 한국어 텍스트 특징 추출 (Analysis)
# =================================================================
print("\n" + "#" * 60)
print("📊 [실습 2] 한국어 텍스트 정량 분석: 평균 길이 측정하기")
print("#" * 60)

def analyze_text(texts):
    """주어진 텍스트 리스트의 평균 글자 수와 특수 문자 비율을 계산합니다."""
    total_char_count = 0
    total_text_count = 0
    punctuation_count = 0
    
    for text in texts:
        if pd.isna(text): continue # 안전 장치
        
        # 한글, 영어, 숫자 등을 포함한 모든 문자를 카운트합니다.
        char_count = len(text)
        total_char_count += char_count
        total_text_count += 1
        
        # 문장 부호(., !, ?, 등)의 개수를 카운트합니다.
        # 유니코드 정규표현식으로 일반적인 구두점 패턴을 찾습니다.
        punctuation_matches = re.findall(r'[.,!?；\s\n]', text) 
        punctuation_count += len(punctuation_matches)

    if total_text_count == 0:
        return 0, 0
        
    avg_length = total_char_count / total_text_count
    avg_punctuation = punctuation_count / total_text_count
    
    return avg_length, avg_punctuation

try:
    # Pandas를 임시로 사용하지만, 실제 구현에서는 필수 외 사용하지 않으므로 경고를 줄입니다.
    import pandas as pd
    avg_length, avg_punctuation = analyze_text(text_samples)
    
    print("✅ 분석 완료! 5개 샘플을 분석한 결과:")
    print(f"  📊 평균 글자 수: {avg_length:.2f} 자 (이 샘플들은 평균적으로 이 정도의 길이를 가집니다.)")
    print(f"  ✨ 평균 구두점 사용 빈도: {avg_punctuation:.2f} 개 (작성자들이 얼마나 자주 문장을 끊었는지 보여줍니다!)")

except NameError:
    print("\nℹ️ 참고: 정량 분석을 위해 'pandas' 라이브러리가 필요할 수 있으나, 지금은 기본 파이썬 기능으로도 충분히 학습할 수 있습니다!")


# =================================================================
# 🏆 실습 3: AI 시뮬레이션 - 프롬프트 생성기 (Creative Use Case)
# =================================================================
print("\n" + "=" * 60)
print("🧠 [실습 3] LLM 프롬프트 엔지니어링 시뮬레이션")
print("==================================================================")

def create_llm_prompt(text, task_name):
    """
    특정 텍스트를 AI가 처리할 수 있는 프롬프트 형태로 가공합니다.
    """
    # 프롬프트의 구조를 명확히 하여 AI의 역할을 정의해주는 것이 중요해요!
    prompt = f"""
[사용자 요청]: 다음 텍스트를 {task_name} 합니다.
[지침]: 결과는 반드시 3개의 번호가 매겨진 목록(bullet list) 형식으로 작성해야 합니다.
[텍스트]:
---
{text}
---
[AI 응답]:
"""
    return prompt.strip()

print("✨ 친절한 튜터 코멘트: AI에게 글을 줄 때는 '이걸 가지고 뭘 해!'라고 지시하는 것이 중요합니다. 이게 바로 '프롬프트 엔지니어링'입니다!")

# 첫 번째 샘플을 골라 분석에 사용해 봅시다.
sample_text = text_samples[0] if text_samples else ""

if sample_text:
    print("\n📘 [첫 번째 샘플을 활용] AI가 '요약'하도록 지시하는 프롬프트 만들기:")
    summary_prompt = create_llm_prompt(sample_text, "핵심 요약")
    print("-" * 20)
    print(summary_prompt)
    
    print("\n📝 [두 번째 시뮬레이션] AI가 '키워드 추출'하도록 지시하는 프롬프트 만들기:")
    keyword_prompt = create_llm_prompt(sample_text, "핵심 키워드 5가지 추출")
    print("-" * 20)
    print(keyword_prompt)

print("\n" + "=" * 60)
print("🎉 수고하셨습니다! 🎉")
print("축하해요! 데이터셋을 로드하고, 분석하고, 심지어 AI에게 명령을 내리는(프롬프트 설계) 과정까지 모두 성공했어요!")
print("다음 단계에서는 이 텍스트를 이용해 실제로 감성을 분석하거나, 개체명을 찾아보는 고급 과정을 진행해 봅시다! 👍")